In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [38]:
data = pd.read_csv("Churn_Modelling_India.csv")
data

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,25687554,Joshi,614,Maharashtra,Female,41,2,0.00,1,1,1,100080.85,1
1,2,25639143,Prasad,610,Karnataka,Female,41,1,83296.82,1,0,1,111976.32,0
2,3,25645631,Gaikwad,499,Maharashtra,Female,43,8,157124.16,3,1,0,112815.53,1
3,4,25563565,Gill,701,Maharashtra,Female,39,1,0.00,2,0,0,94984.36,0
4,5,25564486,Gill,850,Karnataka,Female,43,2,124517.12,1,1,1,79782.27,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,25604568,Sidhu,767,Maharashtra,Male,39,5,0.00,2,1,0,96489.46,0
9996,9997,25765169,Thomas,521,Maharashtra,Male,35,10,57887.16,1,1,1,102342.81,0
9997,9998,25708134,Khanna,707,Maharashtra,Female,35,7,0.00,1,0,1,41460.39,1
9998,9999,25587259,Shukla,775,Delhi,Male,42,3,75416.99,2,1,0,94363.73,1


In [39]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,614,Maharashtra,Female,41,2,0.00,1,1,1,100080.85,1
1,610,Karnataka,Female,41,1,83296.82,1,0,1,111976.32,0
2,499,Maharashtra,Female,43,8,157124.16,3,1,0,112815.53,1
3,701,Maharashtra,Female,39,1,0.00,2,0,0,94984.36,0
4,850,Karnataka,Female,43,2,124517.12,1,1,1,79782.27,0


In [40]:
# Label
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,614,Maharashtra,0,41,2,0.00,1,1,1,100080.85,1
1,610,Karnataka,0,41,1,83296.82,1,0,1,111976.32,0
2,499,Maharashtra,0,43,8,157124.16,3,1,0,112815.53,1
3,701,Maharashtra,0,39,1,0.00,2,0,0,94984.36,0
4,850,Karnataka,0,43,2,124517.12,1,1,1,79782.27,0


In [41]:
# OneHot
onehot_encoder_geo = OneHotEncoder(sparse_output=False)
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']])
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_Delhi', 'Geography_Karnataka', 'Geography_Maharashtra'],
      dtype=object)

In [42]:
geo_encoded_df = pd.DataFrame(geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Delhi,Geography_Karnataka,Geography_Maharashtra
0,614,0,41,2,0.00,1,1,1,100080.85,1,0.0,0.0,1.0
1,610,0,41,1,83296.82,1,0,1,111976.32,0,0.0,1.0,0.0
2,499,0,43,8,157124.16,3,1,0,112815.53,1,0.0,0.0,1.0
3,701,0,39,1,0.00,2,0,0,94984.36,0,0.0,0.0,1.0
4,850,0,43,2,124517.12,1,1,1,79782.27,0,0.0,1.0,0.0


In [43]:
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)
with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

In [44]:
X = data.drop('Exited', axis=1)
y = data['Exited']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=23)

In [45]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [46]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [47]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [48]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid"),
])

In [49]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [50]:
# # Way 1
# model.compile(optimizer="adam",
#               loss="binary_crossentropy",
#               metrics=["accuracy"])

In [51]:
# Way 2
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()

In [52]:
model.compile(optimizer=opt,loss=loss,metrics=["accuracy"])

In [53]:
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = "logs/fit/"+timestamp
tensorflow_callback = TensorBoard(log_dir=log_dir)

In [54]:
# 1000 epochs
# 67 epoch = loss 3.1322
# 68 epoch = loss 3.1322
# 69 epoch = loss 3.1320
# 70 epoch = loss 3.1320
...
# 1000 epoch

early_stopping_callback = EarlyStopping(monitor="val_loss",
              patience=20,
              restore_best_weights=True)

In [55]:
model.fit(X_train, y_train,
          validation_data=(X_test,y_test),
          epochs=1000,
          callbacks=[tensorflow_callback,early_stopping_callback])

Epoch 1/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8395 - loss: 0.3919 - val_accuracy: 0.8575 - val_loss: 0.3483
Epoch 2/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 960us/step - accuracy: 0.8521 - loss: 0.3542 - val_accuracy: 0.8500 - val_loss: 0.3549
Epoch 3/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 996us/step - accuracy: 0.8584 - loss: 0.3481 - val_accuracy: 0.8605 - val_loss: 0.3488
Epoch 4/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 964us/step - accuracy: 0.8576 - loss: 0.3434 - val_accuracy: 0.8545 - val_loss: 0.3390
Epoch 5/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 951us/step - accuracy: 0.8570 - loss: 0.3445 - val_accuracy: 0.8605 - val_loss: 0.3374
Epoch 6/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 936us/step - accuracy: 0.8604 - loss: 0.3399 - val_accuracy: 0.8610 - val_loss: 0.3447
Epoch 7/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8610 - loss: 0.3371 - val_accuracy: 0.8615 - val_loss: 0.3396
Epoch 8/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8624 - loss:

In [56]:
model.save('model.keras')

In [57]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [58]:
# %tensorboard --logdir logs/fit

In [59]:
# tf.keras.backend.clear_session()

In [60]:
%tensorboard --logdir logs/fit --port 6007

Reusing TensorBoard on port 6007 (pid 38332), started 0:02:26 ago. (Use '!kill 38332' to kill it.)